# Presentation Results Builder

Builds compact tables, figures, and Beamer snippets from the proxy strategy experiment outputs. All generated files are written under `outputs/presentation_assets/`.

## 1. Imports and configuration

In [1]:
from pathlib import Path
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ROOT = Path.cwd()
OUT = ROOT / "outputs" / "presentation_assets"
FIG_DIR = OUT / "figures"
TAB_DIR = OUT / "tables"
TEX_DIR = OUT / "latex"
for d in [FIG_DIR, TAB_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

warnings_list = []
generated_tables = []
generated_figures = []
generated_latex = []

PATHS = {
    "baseline_allpairs": ROOT / "outputs/full_runs/full_allpairs_baseline_proxy",
    "wrong_model_allpairs": ROOT / "outputs/full_runs/full_allpairs_wrong_model_proxy",
    "signal_delay_allpairs": ROOT / "outputs/full_runs/full_allpairs_signal_delay_proxy",
    "forced_liq_pair1": ROOT / "outputs/full_runs/final_pair1_forced_liq_both_proxy",
    "sizing_pair1": ROOT / "outputs/full_runs/final_pair1_sizing_sensitivity_proxy",
    "baseline_pair1_fallback": ROOT / "outputs/full_runs/full_pair1_baseline_proxy_v2",
    "wrong_model_pair1_fallback": ROOT / "outputs/full_runs/full_pair1_wrong_model_corrected",
    "forced_liq_fallback": ROOT / "outputs/full_runs/test_forced_liq_both_proxy",
    "sizing_fallback": ROOT / "outputs/full_runs/full_pair1_sizing_sensitivity",
}

REPORTABLE_STRATEGY = "OW_transient_proxy"
PNL_SCALE = 1_000_000.0

## 2. Utility functions

In [2]:
def warn(msg):
    print(f"WARNING: {msg}")
    warnings_list.append(msg)


def safe_read_csv(path, **kwargs):
    if path is None:
        warn("missing CSV path")
        return pd.DataFrame()
    path = Path(path)
    if not path.exists():
        warn(f"missing CSV: {path}")
        return pd.DataFrame()
    try:
        return pd.read_csv(path, **kwargs)
    except Exception as exc:
        warn(f"could not read {path}: {exc}")
        return pd.DataFrame()


def first_existing(paths):
    for path in paths:
        p = Path(path)
        if p.exists():
            return p
    return None


def safe_extract_markdown_section(path, title):
    path = Path(path)
    if not path.exists():
        warn(f"missing markdown report: {path}")
        return ""
    text = path.read_text(errors="ignore")
    pat = re.compile(rf"^##+\s*[^\n]*{re.escape(title)}[^\n]*\n", re.I | re.M)
    m = pat.search(text)
    if not m:
        warn(f"section '{title}' not found in {path}")
        return ""
    nxt = re.search(r"^##\s+", text[m.end():], re.M)
    end = m.end() + nxt.start() if nxt else len(text)
    return text[m.start():end]


def extract_metric(section, label):
    pattern = rf"{re.escape(label)}\s*:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
    m = re.search(pattern, section, re.I)
    return float(m.group(1)) if m else np.nan


def col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def as_millions(s):
    return pd.to_numeric(s, errors="coerce") / PNL_SCALE


def fmt_num(x, digits=2):
    if pd.isna(x):
        return "n/a"
    return f"{float(x):.{digits}f}"


def format_millions(x):
    if pd.isna(x):
        return "n/a"
    return f"{float(x)/PNL_SCALE:.2f}"


def compact_latex(df, path, caption=None, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if df is None or df.empty:
        tex = "% n/a: source data unavailable\n\\begin{tabular}{l}\n n/a \\\\\n\\end{tabular}\n"
    else:
        tex = df.to_latex(index=index, escape=False, na_rep="n/a")
        tex = tex.replace("\\toprule", "\\hline").replace("\\midrule", "\\hline").replace("\\bottomrule", "\\hline")
        if caption:
            tex = f"% {caption}\n" + tex
    path.write_text(tex)
    generated_tables.append(path)
    return path


def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if df is None:
        df = pd.DataFrame()
    df.to_csv(path, index=False)
    generated_tables.append(path)
    return path


def copy_if_exists(src, dst):
    src, dst = Path(src), Path(dst)
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        generated_figures.append(dst)
        return dst
    warn(f"missing file to copy: {src}")
    return None


def save_fig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=250, bbox_inches="tight")
    plt.close()
    generated_figures.append(path)
    return path


def bar_by_pair(df, x_col, y_cols, labels, title, ylabel, path):
    if df.empty or x_col not in df.columns:
        warn(f"cannot plot {path.name}: missing data")
        return None
    x = np.arange(len(df))
    width = 0.8 / max(1, len(y_cols))
    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    for i, (yc, lab) in enumerate(zip(y_cols, labels)):
        if yc not in df.columns:
            continue
        off = (i - (len(y_cols)-1)/2) * width
        ax.bar(x + off, df[yc], width=width, label=lab, color=f"C{i}", alpha=0.85)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(df[x_col].astype(str))
    ax.set_title(title)
    ax.set_xlabel("Pair")
    ax.set_ylabel(ylabel)
    if len(y_cols) > 1:
        ax.legend(frameon=False)
    return save_fig(path)

## 3. Load experiment data

In [3]:
baseline = safe_read_csv(PATHS["baseline_allpairs"] / "all_pairs_strategy_summary.csv")
baseline_fitted = safe_read_csv(PATHS["baseline_allpairs"] / "all_pairs_fitted_proxy_summary.csv")
wrong_model = safe_read_csv(PATHS["wrong_model_allpairs"] / "all_pairs_wrong_model_summary.csv")
signal_delay = safe_read_csv(PATHS["signal_delay_allpairs"] / "all_pairs_stress_summary.csv")
forced_liq = safe_read_csv(first_existing([
    PATHS["forced_liq_pair1"] / "all_pairs_stress_summary.csv",
    PATHS["forced_liq_pair1"] / "pair_1/tables/forced_liquidation_summary.csv",
    PATHS["forced_liq_fallback"] / "all_pairs_stress_summary.csv",
]))
sizing = safe_read_csv(first_existing([
    PATHS["sizing_pair1"] / "all_pairs_sensitivity_summary.csv",
    PATHS["sizing_pair1"] / "pair_1/tables/sizing_sensitivity_summary.csv",
    PATHS["sizing_fallback"] / "all_pairs_sensitivity_summary.csv",
]))

for name, df in [("baseline", baseline), ("baseline_fitted", baseline_fitted), ("wrong_model", wrong_model), ("signal_delay", signal_delay), ("forced_liq", forced_liq), ("sizing", sizing)]:
    print(f"{name}: {df.shape}")

baseline: (11, 31)
baseline_fitted: (11, 13)
wrong_model: (11, 11)
signal_delay: (11, 38)
forced_liq: (4, 63)
sizing: (12, 16)


## 4. Build baseline summaries

In [4]:
base_source = baseline_fitted.copy() if not baseline_fitted.empty else baseline.copy()
if not base_source.empty and "strategy_model" in base_source.columns:
    base_source = base_source[base_source["strategy_model"].eq(REPORTABLE_STRATEGY)].copy()

net_col = col(base_source, ["total_net_pnl_fitted_proxy", "net_pnl_under_ow_regression_eval", "total_net_pnl"])
gross_col = col(base_source, ["total_gross_pnl_fitted_proxy", "total_gross_pnl"])
cost_col = col(base_source, ["total_fitted_proxy_cost", "total_quadratic_impact_cost_normalized", "total_local_proxy_cost"])
sharpe_col = col(base_source, ["daily_sharpe_fitted_proxy", "daily_sharpe"])
dd_col = col(base_source, ["max_drawdown_fitted_proxy", "max_drawdown"])
turnover_col = col(base_source, ["total_turnover_fitted_proxy", "total_signed_volume_turnover", "total_turnover"])
notional_col = col(base_source, ["total_notional_turnover_fitted_proxy", "total_notional_turnover"])
mean_part_col = col(base_source, ["mean_participation_rate_fitted_proxy", "mean_participation_rate"])
max_part_col = col(base_source, ["max_participation_rate_fitted_proxy", "max_participation_rate"])

if base_source.empty or not net_col:
    warn("baseline summary unavailable")
    baseline_pairwise = pd.DataFrame()
    baseline_compact = pd.DataFrame()
    baseline_agg = pd.DataFrame()
else:
    baseline_pairwise = pd.DataFrame({
        "pair_id": base_source.get("pair_id"),
        "net_pnl_m": as_millions(base_source[net_col]),
        "gross_pnl_m": as_millions(base_source[gross_col]) if gross_col else np.nan,
        "cost_m": as_millions(base_source[cost_col]) if cost_col else np.nan,
        "daily_sharpe": pd.to_numeric(base_source[sharpe_col], errors="coerce") if sharpe_col else np.nan,
        "max_drawdown_m": as_millions(base_source[dd_col]) if dd_col else np.nan,
        "turnover_shares_m": as_millions(base_source[turnover_col]) if turnover_col else np.nan,
        "notional_turnover_b": pd.to_numeric(base_source[notional_col], errors="coerce") / 1e9 if notional_col else np.nan,
        "mean_participation_pct": 100 * pd.to_numeric(base_source[mean_part_col], errors="coerce") if mean_part_col else np.nan,
        "max_participation_pct": 100 * pd.to_numeric(base_source[max_part_col], errors="coerce") if max_part_col else np.nan,
    }).sort_values("pair_id")
    save_csv(baseline_pairwise, TAB_DIR / "baseline_summary_pairwise.csv")

    baseline_compact = baseline_pairwise[["pair_id", "net_pnl_m", "daily_sharpe", "max_drawdown_m", "notional_turnover_b", "mean_participation_pct"]].copy()
    baseline_compact.columns = ["Pair", "Net PnL ($m)", "Sharpe", "Max DD ($m)", "Turnover ($bn)", "Mean part. (\\%)"]
    for c in ["Net PnL ($m)", "Sharpe", "Max DD ($m)", "Turnover ($bn)", "Mean part. (\\%)"]:
        baseline_compact[c] = baseline_compact[c].map(lambda x: fmt_num(x, 2))
    compact_latex(baseline_compact, TAB_DIR / "baseline_summary_compact.tex", "All-pairs baseline, OW_transient_proxy")

    baseline_agg = pd.DataFrame({
        "Statistic": ["Mean net PnL", "Median net PnL", "Min net PnL", "Max net PnL", "Mean Sharpe", "Mean max drawdown", "Total net PnL"],
        "Value": [
            f"{baseline_pairwise['net_pnl_m'].mean():.2f} $m",
            f"{baseline_pairwise['net_pnl_m'].median():.2f} $m",
            f"{baseline_pairwise['net_pnl_m'].min():.2f} $m",
            f"{baseline_pairwise['net_pnl_m'].max():.2f} $m",
            f"{baseline_pairwise['daily_sharpe'].mean():.2f}",
            f"{baseline_pairwise['max_drawdown_m'].mean():.2f} $m",
            f"{baseline_pairwise['net_pnl_m'].sum():.2f} $m",
        ]
    })
    compact_latex(baseline_agg, TAB_DIR / "baseline_aggregate_stats.tex", "Aggregate baseline statistics")

    p1 = baseline_pairwise[baseline_pairwise["pair_id"].astype(str).eq("1")].head(1)
    if not p1.empty:
        baseline_pair1_detail = pd.DataFrame({
            "Metric": ["Gross PnL", "Fitted proxy cost", "Net PnL", "Daily Sharpe", "Max drawdown", "Mean participation"],
            "Value": [
                f"{p1['gross_pnl_m'].iloc[0]:.2f} $m",
                f"{p1['cost_m'].iloc[0]:.2f} $m",
                f"{p1['net_pnl_m'].iloc[0]:.2f} $m",
                f"{p1['daily_sharpe'].iloc[0]:.2f}",
                f"{p1['max_drawdown_m'].iloc[0]:.2f} $m",
                f"{p1['mean_participation_pct'].iloc[0]:.2f}\\%",
            ]
        })
    else:
        baseline_pair1_detail = pd.DataFrame()
    compact_latex(baseline_pair1_detail, TAB_DIR / "baseline_pair1_detail.tex", "Pair 1 baseline detail, fitted proxy PnL")

    bar_by_pair(baseline_pairwise, "pair_id", ["net_pnl_m"], ["Net PnL"], "Baseline net PnL by pair", "Net PnL ($m)", FIG_DIR / "baseline_net_pnl_by_pair.png")
    bar_by_pair(baseline_pairwise, "pair_id", ["daily_sharpe"], ["Daily Sharpe"], "Baseline Sharpe by pair", "Daily Sharpe", FIG_DIR / "baseline_sharpe_by_pair.png")

wealth_src = first_existing([
    PATHS["baseline_allpairs"] / "pair_1/figures/pair_cumulative_wealth_reportable_strategy.png",
    PATHS["baseline_pair1_fallback"] / "pair_1/figures/pair_cumulative_wealth_reportable_strategy.png",
    PATHS["forced_liq_pair1"] / "pair_1/figures/pair_cumulative_wealth_reportable_strategy.png",
])
if wealth_src:
    copy_if_exists(wealth_src, FIG_DIR / "ow_proxy_wealth_pair1.png")
else:
    warn("pair 1 wealth curve not found")

baseline_pair1 = baseline_pairwise[baseline_pairwise["pair_id"].astype(str).eq("1")].copy() if not baseline_pairwise.empty else pd.DataFrame()
display(baseline_pairwise.head())
display(baseline_agg)

,pair_id,net_pnl_m,gross_pnl_m,cost_m,daily_sharpe,max_drawdown_m,turnover_shares_m,notional_turnover_b,mean_participation_pct,max_participation_pct
0,1,32.522365,33.023385,0.501021,1.881180,-1.670632,282.617226,42.786914,0.016061,1.0
1,2,42.379923,42.985011,0.605087,1.837411,-0.871076,388.692970,49.946290,0.019046,1.0
2,3,32.491074,33.125854,0.634780,1.773233,-0.915906,392.948281,43.886273,0.018442,1.0
3,4,54.365250,55.108904,0.743654,2.107079,-0.836233,451.476267,68.202046,0.023099,1.0
4,5,60.123359,60.793224,0.669865,1.580381,-0.833752,319.581783,72.088293,0.020053,1.0


,Statistic,Value
0,Mean net PnL,48.87 $m
1,Median net PnL,38.85 $m
2,Min net PnL,28.15 $m
3,Max net PnL,105.00 $m
4,Mean Sharpe,2.27
5,Mean max drawdown,-0.86 $m
6,Total net PnL,537.57 $m


## 5. Extract alpha diagnostics

In [5]:
alpha_report = first_existing([
    PATHS["baseline_pair1_fallback"] / "reports/integrated_report.md",
    PATHS["baseline_allpairs"] / "reports/integrated_report.md",
])
section = safe_extract_markdown_section(alpha_report, "Alpha Diagnostics") if alpha_report else ""
metrics = {
    "alpha_mean_bps": extract_metric(section, "alpha_mean_bps"),
    "alpha_std_bps": extract_metric(section, "alpha_std_bps"),
    "corr(alpha, future_return_h)": extract_metric(section, "corr(alpha, future_return_h)"),
    "corr(trade, alpha)": extract_metric(section, "corr(trade, alpha)"),
    "corr(position_after, alpha)": extract_metric(section, "corr(position_after, alpha)"),
    "share_sign_trade_matches_alpha": extract_metric(section, "share_sign_trade_matches_alpha"),
    "gross alpha capture": extract_metric(section, "gross alpha capture"),
}
alpha_metrics = pd.DataFrame([{"metric": k, "value": v} for k, v in metrics.items()])
save_csv(alpha_metrics, TAB_DIR / "alpha_metrics_pair1.csv")
alpha_tex = alpha_metrics.copy()
alpha_tex["value"] = alpha_tex.apply(lambda r: format_millions(r["value"]) if r["metric"] == "gross alpha capture" else fmt_num(r["value"], 3), axis=1)
alpha_tex["metric"] = alpha_tex["metric"].replace({
    "alpha_mean_bps": "$\\mu(\\alpha)$ bps",
    "alpha_std_bps": "$\\sigma(\\alpha)$ bps",
    "corr(alpha, future_return_h)": "$\\rho(\\alpha,r_{t+H})$",
    "corr(trade, alpha)": "$\\rho(q,\\alpha)$",
    "corr(position_after, alpha)": "$\\rho(Q^+,\\alpha)$",
    "share_sign_trade_matches_alpha": "$\\Pr[\\operatorname{sign}(q)=\\operatorname{sign}(\\alpha)]$",
    "gross alpha capture": "Gross alpha ($m)",
})
alpha_tex.columns = ["Metric", "Value"]
compact_latex(alpha_tex, TAB_DIR / "alpha_metrics_pair1.tex", "Pair 1 alpha diagnostics")
display(alpha_tex)

,Metric,Value
0,$\mu(\alpha)$ bps,0.001
1,$\sigma(\alpha)$ bps,0.232
2,"$\rho(\alpha,r_{t+H})$",0.190
3,"$\rho(q,\alpha)$",0.490
4,"$\rho(Q^+,\alpha)$",0.592
5,$\Pr[\operatorname{sign}(q)=\operatorname{sign...,0.858
6,Gross alpha ($m),33.02


## 6. Build wrong-model tables/figures

In [6]:
wm_pair1_matrix_file = first_existing([
    PATHS["wrong_model_allpairs"] / "pair_1/stress/wrong_model_matrix.csv",
    PATHS["wrong_model_pair1_fallback"] / "pair_1/stress/wrong_model_matrix.csv",
])
wm_matrix_raw = safe_read_csv(wm_pair1_matrix_file) if wm_pair1_matrix_file else pd.DataFrame()

if not wm_matrix_raw.empty and {"assumed_model", "evaluator_model", "net_pnl"}.issubset(wm_matrix_raw.columns):
    wm_pivot = wm_matrix_raw.pivot_table(index="assumed_model", columns="evaluator_model", values="net_pnl", aggfunc="first") / PNL_SCALE
    wm_pivot = wm_pivot.reindex(index=["OW_transient", "reduced_form"], columns=["OW_transient", "reduced_form"])
else:
    pair1 = wrong_model[wrong_model["pair_id"].astype(str).eq("1")].head(1) if not wrong_model.empty and "pair_id" in wrong_model else pd.DataFrame()
    if pair1.empty:
        warn("wrong-model pair 1 matrix unavailable")
        wm_pivot = pd.DataFrame()
    else:
        r = pair1.iloc[0]
        wm_pivot = pd.DataFrame(
            [[r.get("ow_true_correct_model_pnl", np.nan), r.get("rf_true_wrong_model_pnl", np.nan)],
             [r.get("ow_true_wrong_model_pnl", np.nan), r.get("rf_true_correct_model_pnl", np.nan)]],
            index=["OW_transient", "reduced_form"],
            columns=["OW_transient", "reduced_form"],
        ) / PNL_SCALE

wm_tex = wm_pivot.copy()
if not wm_tex.empty:
    wm_tex.index = ["OW proxy", "RF proxy"]
    wm_tex.columns = ["OW eval", "RF eval"]
    wm_tex = wm_tex.applymap(lambda x: fmt_num(x, 2))
compact_latex(wm_tex.reset_index().rename(columns={"index": "Assumed strategy"}), TAB_DIR / "wrong_model_matrix_pair1.tex", "Pair 1 net PnL ($m)")

if wrong_model.empty:
    wrong_summary = pd.DataFrame()
else:
    wrong_summary = wrong_model.copy()
    for c in ["wrong_model_loss_rf_true", "wrong_model_loss_ow_true", "net_pnl_difference_rf_minus_ow"]:
        if c in wrong_summary.columns:
            wrong_summary[c + "_m"] = wrong_summary[c] / PNL_SCALE
    keep = [c for c in ["pair_id", "wrong_model_loss_rf_true_m", "wrong_model_loss_ow_true_m", "net_pnl_difference_rf_minus_ow_m"] if c in wrong_summary.columns]
    wrong_summary = wrong_summary[keep].sort_values("pair_id")
    wrong_tex = wrong_summary.copy()
    wrong_tex.columns = ["Pair", "Loss RF-true ($m)", "Loss OW-true ($m)", "RF-OW diff. ($m)"][:len(wrong_tex.columns)]
    for c in wrong_tex.columns[1:]:
        wrong_tex[c] = wrong_tex[c].map(lambda x: fmt_num(x, 2))
    compact_latex(wrong_tex, TAB_DIR / "wrong_model_allpairs_summary.tex", "Wrong-model loss = wrong strategy PnL - correct strategy PnL")
    bar_by_pair(wrong_summary, "pair_id", ["wrong_model_loss_rf_true_m", "wrong_model_loss_ow_true_m"], ["RF true", "OW true"], "Wrong-model losses by pair", "Loss ($m)", FIG_DIR / "wrong_model_losses_by_pair.png")

display(wm_tex)
display(wrong_summary.head() if 'wrong_summary' in locals() else pd.DataFrame())

/var/folders/95/b7t5chdd7sx2c8x5m_kmybrh0000gn/T/ipykernel_13343/2473861474.py:28: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wm_tex = wm_tex.applymap(lambda x: fmt_num(x, 2))


,OW eval,RF eval
OW proxy,32.40,32.35
RF proxy,30.04,30.34


,pair_id,wrong_model_loss_rf_true_m,wrong_model_loss_ow_true_m,net_pnl_difference_rf_minus_ow_m
0,1,2.006388,-2.358823,-2.055436
1,2,0.477912,-0.967565,-0.680541
2,3,-1.496821,-0.616801,1.403957
3,4,-0.896545,0.311866,1.151229
4,5,-9.802700,9.391183,10.167660


## 7. Build signal-delay tables/figures

In [7]:
if signal_delay.empty or baseline.empty:
    warn("signal-delay or baseline data missing")
    sig = pd.DataFrame()
else:
    b = baseline[["pair_id", "total_net_pnl"]].rename(columns={"total_net_pnl": "baseline_net_pnl"}).copy()
    delayed_col = col(signal_delay, ["net_pnl_under_ow_regression_eval", "total_net_pnl"])
    s = signal_delay[["pair_id", delayed_col]].rename(columns={delayed_col: "delayed_net_pnl"}).copy()
    sig = b.merge(s, on="pair_id", how="inner")
    sig["degradation"] = sig["delayed_net_pnl"] - sig["baseline_net_pnl"]
    sig["degradation_pct"] = sig["degradation"] / sig["baseline_net_pnl"].replace(0, np.nan)
    sig_out = pd.DataFrame({
        "pair_id": sig["pair_id"],
        "baseline_net_pnl_m": sig["baseline_net_pnl"] / PNL_SCALE,
        "delayed_net_pnl_m": sig["delayed_net_pnl"] / PNL_SCALE,
        "degradation_m": sig["degradation"] / PNL_SCALE,
        "degradation_pct": 100 * sig["degradation_pct"],
    }).sort_values("pair_id")
    save_csv(sig_out, TAB_DIR / "signal_delay_summary.csv")
    sig_tex = sig_out.copy()
    sig_tex.columns = ["Pair", "Base ($m)", "Delayed ($m)", "$\\Delta$ ($m)", "$\\Delta$ (\\%)"]
    for c in sig_tex.columns[1:4]:
        sig_tex[c] = sig_tex[c].map(lambda x: fmt_num(x, 2))
    sig_tex["$\\Delta$ (\\%)"] = sig_tex["$\\Delta$ (\\%)"].map(lambda x: fmt_num(x, 1))
    compact_latex(sig_tex, TAB_DIR / "signal_delay_summary.tex", "Signal delay summary")
    sig_summary = pd.DataFrame({
        "Statistic": ["Mean degradation", "Median degradation", "Worst pair", "Total baseline", "Total delayed", "Total degradation"],
        "Value": [
            f"{100*sig['degradation_pct'].mean():.1f}\\%",
            f"{100*sig['degradation_pct'].median():.1f}\\%",
            f"{100*sig['degradation_pct'].min():.1f}\\%",
            f"{sig['baseline_net_pnl'].sum()/PNL_SCALE:.2f} $m",
            f"{sig['delayed_net_pnl'].sum()/PNL_SCALE:.2f} $m",
            f"{sig['degradation'].sum()/PNL_SCALE:.2f} $m",
        ]
    })
    compact_latex(sig_summary, TAB_DIR / "signal_delay_aggregate.tex", "Signal delay aggregate degradation")
    bar_by_pair(sig_out, "pair_id", ["degradation_m"], ["Degradation"], "Signal-delay degradation by pair", "Delayed - baseline ($m)", FIG_DIR / "signal_delay_degradation_by_pair.png")

display(sig.head() if not sig.empty else sig)

,pair_id,baseline_net_pnl,delayed_net_pnl,degradation,degradation_pct
0,1,3.252236e+07,2.183466e+07,-1.068771e+07,-0.328626
1,2,4.237992e+07,2.953904e+07,-1.284088e+07,-0.302994
2,3,3.249107e+07,2.324384e+07,-9.247237e+06,-0.284609
3,4,5.436525e+07,3.579431e+07,-1.857094e+07,-0.341596
4,5,6.012336e+07,4.200689e+07,-1.811647e+07,-0.301322


## 8. Build forced-liquidation tables/figures

In [8]:
if forced_liq.empty:
    fl_out = pd.DataFrame()
    warn("forced-liquidation summary unavailable")
else:
    fl = forced_liq.copy()
    ow_pnl = col(fl, ["net_pnl_under_ow_regression_eval", "stress_net_pnl_ow_eval", "total_net_pnl"])
    deg = col(fl, ["degradation_ow_eval"])
    fl_out = pd.DataFrame({
        "trigger": fl.get("liquidation_trigger_mode", "n/a"),
        "mode": fl.get("liquidation_mode", "n/a"),
        "events": fl.get("number_of_liquidation_events", np.nan),
        "event_rate_pct": 100 * pd.to_numeric(fl.get("liquidation_event_rate_realized", np.nan), errors="coerce"),
        "ow_eval_pnl_m": as_millions(fl[ow_pnl]) if ow_pnl else np.nan,
        "degradation_m": as_millions(fl[deg]) if deg else np.nan,
        "cap_violation_pct": 100 * pd.to_numeric(fl.get("hard_block_cap_violation_rate", np.nan), errors="coerce"),
        "residual_after_first": fl.get("number_with_residual_after_first_liquidation", np.nan),
    })
    save_csv(fl_out, TAB_DIR / "forced_liq_summary.csv")
    fl_tex = fl_out.copy()
    fl_tex.columns = ["Trigger", "Mode", "Events", "Event (\\%)", "OW PnL ($m)", "$\\Delta$ ($m)", "Cap viol. (\\%)", "Residual"]
    for c in ["Event (\\%)", "OW PnL ($m)", "$\\Delta$ ($m)", "Cap viol. (\\%)"]:
        fl_tex[c] = fl_tex[c].map(lambda x: fmt_num(x, 2 if "$m" in c or "PnL" in c else 1))
    compact_latex(fl_tex, TAB_DIR / "forced_liq_summary.tex", "Forced liquidation, pair 1")

    fl_plot = fl_out.copy()
    fl_plot["label"] = fl_plot["trigger"].astype(str).str.replace("_daily", "", regex=False) + "\n" + fl_plot["mode"].astype(str).str.replace("capped_with_residual", "capped", regex=False).str.replace("hard_block", "hard", regex=False)
    fig, ax = plt.subplots(figsize=(7.4, 3.8))
    ax.bar(np.arange(len(fl_plot)), fl_plot["ow_eval_pnl_m"], color="C0", alpha=0.85)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(np.arange(len(fl_plot)))
    ax.set_xticklabels(fl_plot["label"], rotation=0)
    ax.set_title("Forced-liquidation net PnL by mode")
    ax.set_ylabel("OW-eval net PnL ($m)")
    save_fig(FIG_DIR / "forced_liq_net_pnl_by_mode.png")

display(fl_out)

,trigger,mode,events,event_rate_pct,ow_eval_pnl_m,degradation_m,cap_violation_pct,residual_after_first
0,deterministic_daily,hard_block,380,100.000000,21.354557,-11.044065,55.263158,0
1,deterministic_daily,capped_with_residual,380,100.000000,21.880353,-10.518269,0.000000,210
2,probabilistic_daily,hard_block,33,8.684211,30.636359,-1.762263,51.515152,0
3,probabilistic_daily,capped_with_residual,33,8.684211,30.624553,-1.774068,0.000000,17


## 9. Build sizing-sensitivity tables/figures

In [9]:
if sizing.empty:
    sz = pd.DataFrame()
    warn("sizing sensitivity unavailable")
else:
    sz = sizing.copy()
    pnl_col = col(sz, ["net_pnl_under_ow_regression_eval", "total_net_pnl"])
    turnover_col = col(sz, ["total_turnover", "total_signed_volume_turnover"])
    sz["net_pnl_m"] = as_millions(sz[pnl_col]) if pnl_col else np.nan
    sz["turnover_m"] = as_millions(sz[turnover_col]) if turnover_col else np.nan
    pivot_pnl = sz.pivot_table(index="target_impact_scale", columns="max_participation_rate", values="net_pnl_m", aggfunc="first").sort_index()
    save_csv(pivot_pnl.reset_index(), TAB_DIR / "sizing_sensitivity_summary.csv")
    pnl_tex = pivot_pnl.copy().applymap(lambda x: fmt_num(x, 2))
    pnl_tex.index.name = "Target scale"
    pnl_tex.columns = [f"{100*c:.1f}\\% cap" for c in pnl_tex.columns]
    compact_latex(pnl_tex.reset_index(), TAB_DIR / "sizing_sensitivity_summary.tex", "Net PnL ($m) by target scale and participation cap")

    fig, ax = plt.subplots(figsize=(7.2, 3.9))
    for cap, grp in sz.sort_values(["max_participation_rate", "target_impact_scale"]).groupby("max_participation_rate"):
        ax.plot(grp["target_impact_scale"], grp["net_pnl_m"], marker="o", lw=1.8, label=f"cap {100*cap:.1f}%")
    ax.set_title("Sizing sensitivity: net PnL")
    ax.set_xlabel("Target impact scale")
    ax.set_ylabel("OW-eval net PnL ($m)")
    ax.legend(frameon=False)
    save_fig(FIG_DIR / "sizing_sensitivity_net_pnl.png")

    fig, ax = plt.subplots(figsize=(7.2, 3.9))
    for cap, grp in sz.sort_values(["max_participation_rate", "target_impact_scale"]).groupby("max_participation_rate"):
        ax.plot(grp["target_impact_scale"], grp["turnover_m"], marker="o", lw=1.8, label=f"cap {100*cap:.1f}%")
    ax.set_title("Sizing sensitivity: turnover")
    ax.set_xlabel("Target impact scale")
    ax.set_ylabel("Turnover (m shares)")
    ax.legend(frameon=False)
    save_fig(FIG_DIR / "sizing_sensitivity_turnover.png")

display(pivot_pnl if 'pivot_pnl' in locals() else pd.DataFrame())

/var/folders/95/b7t5chdd7sx2c8x5m_kmybrh0000gn/T/ipykernel_13343/3736778430.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pnl_tex = pivot_pnl.copy().applymap(lambda x: fmt_num(x, 2))


max_participation_rate,0.005,0.010,0.020
target_impact_scale,,,
0.10,3.652427,3.644779,3.644779
0.25,9.762629,9.732344,9.732344
0.50,18.721876,18.705303,18.705303
1.00,32.382040,32.398622,32.398622


## 10. Generate LaTeX Beamer snippets

In [10]:
slides = r"""
% Auto-generated by presentation_results_builder.ipynb

\begin{frame}{2.4 Alpha signal}
\begin{block}{Construction}
Synthetic short-horizon alpha with $H=5$m and $\rho=0.10$:
\[
\alpha_t^{bps} \approx 10^4\mathbb{E}\left[\frac{S_{t+H}-S_t}{S_t}\mid \mathcal F_t\right].
\]
\end{block}
\begin{columns}[T]
\column{0.48\textwidth}
\begin{itemize}
  \item Positive out-of-sample predictive content.
  \item Trades align with alpha direction.
  \item Gross alpha capture is positive before impact costs.
\end{itemize}
\column{0.52\textwidth}
\scriptsize\input{outputs/presentation_assets/tables/alpha_metrics_pair1.tex}
\end{columns}
\end{frame}

\begin{frame}{2.5 OW transient proxy strategy}
\begin{block}{Local proxy objective}
\[
\max_{q_t}\; q_t\alpha_t - \kappa_t q_t^2 - \gamma(Q_{t^-}+q_t)^2
\]
\end{block}
\begin{columns}[T]
\column{0.48\textwidth}
\begin{itemize}
  \item Reportable strategy: \texttt{OW\_transient\_proxy}.
  \item Participation cap: $|q_t|\leq 1\%$ volume.
  \item Inventory cap: $|Q_t|\leq 5\%$ ADV.
\end{itemize}
\column{0.52\textwidth}
\scriptsize\input{outputs/presentation_assets/tables/baseline_pair1_detail.tex}
\end{columns}
\end{frame}

\begin{frame}{2.5 OW proxy wealth: pair 1}
\begin{center}
\includegraphics[width=0.88\textwidth]{outputs/presentation_assets/figures/ow_proxy_wealth_pair1.png}
\end{center}
\end{frame}

\begin{frame}{2.5 Baseline all-pairs results}
\begin{columns}[T]
\column{0.42\textwidth}
\scriptsize\input{outputs/presentation_assets/tables/baseline_aggregate_stats.tex}
\column{0.58\textwidth}
\includegraphics[width=\textwidth]{outputs/presentation_assets/figures/baseline_net_pnl_by_pair.png}
\vspace{0.1cm}
\includegraphics[width=\textwidth]{outputs/presentation_assets/figures/baseline_sharpe_by_pair.png}
\end{columns}
\end{frame}

\begin{frame}{2.5 Reduced-form proxy and model comparison}
\begin{block}{Same proxy, richer slope}
Reduced-form proxy uses the same myopic quadratic mechanism with marginal impact slope from the richer regression specification.
\end{block}
\begin{columns}[T]
\column{0.48\textwidth}
\begin{itemize}
  \item Compare model-consistent baselines against misspecified strategy paths.
  \item Correct RF: RF proxy under RF evaluator.
  \item Correct OW: OW proxy under OW evaluator.
\end{itemize}
\column{0.52\textwidth}
\scriptsize\input{outputs/presentation_assets/tables/wrong_model_matrix_pair1.tex}
\end{columns}
\end{frame}

\begin{frame}{Stress test: wrong model}
\begin{columns}[T]
\column{0.50\textwidth}
\scriptsize\input{outputs/presentation_assets/tables/wrong_model_allpairs_summary.tex}
\column{0.50\textwidth}
\includegraphics[width=\textwidth]{outputs/presentation_assets/figures/wrong_model_losses_by_pair.png}
\end{columns}
\vspace{0.2cm}
\small Misspecification is path-dependent: the wrong model can outperform in some pairs because it changes aggressiveness and exposure, but losses are unstable and asymmetric.
\end{frame}

\begin{frame}{Stress test: signal delay}
\begin{columns}[T]
\column{0.52\textwidth}
\scriptsize\input{outputs/presentation_assets/tables/signal_delay_summary.tex}
\column{0.48\textwidth}
\includegraphics[width=\textwidth]{outputs/presentation_assets/figures/signal_delay_degradation_by_pair.png}
\end{columns}
\vspace{0.2cm}
\small One-minute signal delay materially reduces alpha capture, confirming the short-horizon nature of the signal.
\end{frame}

\begin{frame}{Stress test: forced liquidation}
\begin{columns}[T]
\column{0.58\textwidth}
\scriptsize\input{outputs/presentation_assets/tables/forced_liq_summary.tex}
\column{0.42\textwidth}
\includegraphics[width=\textwidth]{outputs/presentation_assets/figures/forced_liq_net_pnl_by_mode.png}
\end{columns}
\vspace{0.15cm}
\small Deterministic daily liquidation is a severe systematic shock; probabilistic liquidation is more realistic. Hard block is an upper-bound shock; capped residual is operationally feasible.
\end{frame}

\begin{frame}{Stress test: sizing sensitivity}
\begin{columns}[T]
\column{0.48\textwidth}
\includegraphics[width=\textwidth]{outputs/presentation_assets/figures/sizing_sensitivity_net_pnl.png}
\column{0.48\textwidth}
\includegraphics[width=\textwidth]{outputs/presentation_assets/figures/sizing_sensitivity_turnover.png}
\end{columns}
\vspace{0.2cm}
\small Sizing dominates economics: higher participation and target impact increase turnover and may improve gross alpha capture, but also increase impact-cost exposure and clipping.
\end{frame}
"""

tex_path = TEX_DIR / "presentation_slides_snippets.tex"
tex_path.write_text(slides.strip() + "\n")
generated_latex.append(tex_path)
print(tex_path)

/Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/presentation_assets/latex/presentation_slides_snippets.tex


## 11. Final checklist

In [11]:
required = [
    TAB_DIR / "baseline_summary_pairwise.csv",
    TAB_DIR / "baseline_summary_compact.tex",
    TAB_DIR / "wrong_model_matrix_pair1.tex",
    TAB_DIR / "wrong_model_allpairs_summary.tex",
    TAB_DIR / "signal_delay_summary.tex",
    TAB_DIR / "forced_liq_summary.tex",
    TAB_DIR / "sizing_sensitivity_summary.tex",
    FIG_DIR / "baseline_net_pnl_by_pair.png",
    FIG_DIR / "baseline_sharpe_by_pair.png",
    FIG_DIR / "signal_delay_degradation_by_pair.png",
    FIG_DIR / "wrong_model_losses_by_pair.png",
    FIG_DIR / "forced_liq_net_pnl_by_mode.png",
    FIG_DIR / "sizing_sensitivity_net_pnl.png",
    FIG_DIR / "sizing_sensitivity_turnover.png",
    TEX_DIR / "presentation_slides_snippets.tex",
]

print("Generated figures:")
for p in sorted(set(generated_figures)):
    print(" -", p.relative_to(ROOT))
print("\nGenerated tables:")
for p in sorted(set(generated_tables)):
    print(" -", p.relative_to(ROOT))
print("\nLaTeX snippets:")
for p in sorted(set(generated_latex)):
    print(" -", p.relative_to(ROOT))

missing_required = [p for p in required if not p.exists()]
if missing_required:
    print("\nMissing required outputs:")
    for p in missing_required:
        print(" -", p.relative_to(ROOT))
else:
    print("\nAll required outputs exist.")

if warnings_list:
    print("\nWarnings:")
    for w in warnings_list:
        print(" -", w)
else:
    print("\nWarnings: none")

Generated figures:
 - outputs/presentation_assets/figures/baseline_net_pnl_by_pair.png
 - outputs/presentation_assets/figures/baseline_sharpe_by_pair.png
 - outputs/presentation_assets/figures/forced_liq_net_pnl_by_mode.png
 - outputs/presentation_assets/figures/ow_proxy_wealth_pair1.png
 - outputs/presentation_assets/figures/signal_delay_degradation_by_pair.png
 - outputs/presentation_assets/figures/sizing_sensitivity_net_pnl.png
 - outputs/presentation_assets/figures/sizing_sensitivity_turnover.png
 - outputs/presentation_assets/figures/wrong_model_losses_by_pair.png

Generated tables:
 - outputs/presentation_assets/tables/alpha_metrics_pair1.csv
 - outputs/presentation_assets/tables/alpha_metrics_pair1.tex
 - outputs/presentation_assets/tables/baseline_aggregate_stats.tex
 - outputs/presentation_assets/tables/baseline_pair1_detail.tex
 - outputs/presentation_assets/tables/baseline_summary_compact.tex
 - outputs/presentation_assets/tables/baseline_summary_pairwise.csv
 - outputs/pres